In [5]:
!pip -q uninstall -y torch torchvision torchaudio
!pip -q cache purge

!pip -q install --index-url https://download.pytorch.org/whl/cu121 \
  torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 118.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 125.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 75.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 59.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 101.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 9.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 15.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 10.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
!pip -q install -U transformers

import torch, torchvision, torchaudio
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("torchaudio:", torchaudio.__version__)

from transformers import pipeline
clf = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
print(clf("This is fine."))


torch: 2.5.1+cu121
torchvision: 0.20.1+cu121
torchaudio: 2.5.1+cu121


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9998524188995361}]


In [5]:


import re
import tarfile
import urllib.request
from pathlib import Path
from typing import List, Tuple, Dict
from transformers import pipeline

IMDB_URL = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"

def download_and_extract_imdb(data_dir: str = "/content/data") -> Path:
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)

    tgz_path = data_dir / "aclImdb_v1.tar.gz"
    extract_root = data_dir / "aclImdb"

    if not extract_root.exists():
        if not tgz_path.exists():
            print(f"Downloading to {tgz_path} ...")
            urllib.request.urlretrieve(IMDB_URL, tgz_path)
        print(f"Extracting to {data_dir} ...")
        with tarfile.open(tgz_path, "r:gz") as tf:
            tf.extractall(path=data_dir)

    return extract_root

_word_re = re.compile(r"[A-Za-z0-9']+")
def tokenize(text: str) -> List[str]:
    return _word_re.findall(text.lower())

def iter_review_texts(aclImdb_root: Path, split: str = "train", labels=("pos", "neg")):
    for label in labels:
        folder = aclImdb_root / split / label
        for fp in folder.glob("*.txt"):
            yield fp.read_text(encoding="utf-8", errors="ignore")

# --- Sentiment scoring ---
sent = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

def p_pos_from_result(res) -> float:

    if res["label"].upper().startswith("POS"):
        return float(res["score"])
    else:
        return 1.0 - float(res["score"])

def is_neutral(prefix: str, neutral_band: float = 0.10) -> Tuple[bool, float]:
    res = sent(prefix)[0]
    p_pos = p_pos_from_result(res)
    return (abs(p_pos - 0.5) <= neutral_band), p_pos

def collect_neutral_unique_prefixes(
    texts: List[str],
    target: int = 100,
    start_n: int = 5,
    max_n: int = 20,
    neutral_band: float = 0.10,
    max_texts_to_scan: int = 20000,
) -> Tuple[List[Tuple[int, str, float]], Dict[int, int]]:
    tokenized = [tokenize(t) for t in texts[:max_texts_to_scan]]

    selected: List[Tuple[int, str, float]] = []
    seen = set()
    counts: Dict[int, int] = {}

    for n in range(start_n, max_n + 1):
        if len(selected) >= target:
            break

        added = 0
        for toks in tokenized:
            if len(selected) >= target:
                break
            if len(toks) < n:
                continue

            pref = " ".join(toks[:n])
            if pref in seen:
                continue

            ok, p_pos = is_neutral(pref, neutral_band=neutral_band)
            if ok:
                seen.add(pref)
                selected.append((n, pref, p_pos))
                added += 1

        counts[n] = added

    return selected, counts

# ---- Run ----
root = download_and_extract_imdb("/content/data")
texts = list(iter_review_texts(root, split="train", labels=("pos","neg")))

prefixes, counts = collect_neutral_unique_prefixes(
    texts,
    target=100,
    start_n=5,
    max_n=20,
    neutral_band=0.05,
    max_texts_to_scan=20000
)

print("Neutral prefixes added by n:")
for n in sorted(counts):
    if counts[n] > 0:
        print(f"  {n} words: {counts[n]}")

print(f"\nTotal collected: {len(prefixes)}\n")

for i, (n, p, ppos) in enumerate(prefixes, start=1):
    print(f"{i:3d}. [{n}w | p_pos={ppos:.3f}] {p}")


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Neutral prefixes added by n:
  5 words: 100

Total collected: 100

  1. [5w | p_pos=0.527] the haunted world of edward
  2. [5w | p_pos=0.459] reed hadley makes a better
  3. [5w | p_pos=0.481] although i was born in
  4. [5w | p_pos=0.523] although super mario 64 isn't
  5. [5w | p_pos=0.497] first off i must say
  6. [5w | p_pos=0.509] believe it or not at
  7. [5w | p_pos=0.500] re pro jury br br
  8. [5w | p_pos=0.476] in the trivia section for
  9. [5w | p_pos=0.502] let me first state that
 10. [5w | p_pos=0.524] i read some previous comments
 11. [5w | p_pos=0.492] i couldn't help but feel
 12. [5w | p_pos=0.525] i thought this was an
 13. [5w | p_pos=0.482] watch on the rhine is
 14. [5w | p_pos=0.541] human tornado 1976 is in
 15. [5w | p_pos=0.502] apart from the usual stereotypes
 16. [5w | p_pos=0.513] i've only ever seen this
 17. [5w | p_pos=0.523] you don't need to read
 18. [5w | p_pos=0.511] after i watched the films
 19. [5w | p_pos=0.503] i can't believe currently th

In [6]:
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
from pathlib import Path

df = pd.DataFrame(prefixes, columns=["n_words", "prefix", "p_pos"])

out_path = Path("/content/drive/MyDrive/neutral_prefixes.csv")
df.to_csv(out_path, index=False)

print("Saved to:", out_path)


Mounted at /content/drive
Saved to: /content/drive/MyDrive/neutral_prefixes.csv
